# 05 — Celeb-DF Cross-Dataset Preparation

This notebook creates the fixed Celeb-DF v2 pilot dataset used by the separate Section 5 evaluation notebooks.

It performs a fresh build when the pilot does not exist and safely reuses the saved manifest, frames and face crops on subsequent runs. It does **not** load experiment checkpoints, run inference or calculate model metrics.


## 1. Build controls and rerun safety

Normal first build: leave `PREPARE_CELEBDF_PILOT = True` and `FORCE_REBUILD = False`.

Normal rerun: the saved manifest is loaded and complete outputs are skipped.

Forced rebuilding is intentionally difficult: both the Boolean flag and exact confirmation text are required.


In [1]:
INSTALL_DEPENDENCIES = True
PREPARE_CELEBDF_PILOT = True
AUDIT_CELEBDF_PILOT = True

FORCE_REBUILD = False
REBUILD_CONFIRMATION = ""
REQUIRED_REBUILD_CONFIRMATION = "REBUILD_CELEBDF_PILOT"

SEED = 42
PILOT_REAL_VIDEOS = 20
PILOT_FAKE_VIDEOS = 20
FRAMES_PER_VIDEO = 10
FACE_MARGIN_RATIO = 0.15
FACE_SIZE = 224

if FORCE_REBUILD:
    assert REBUILD_CONFIRMATION == REQUIRED_REBUILD_CONFIRMATION, (
        "Forced rebuilding requires REBUILD_CONFIRMATION = "
        f"'{REQUIRED_REBUILD_CONFIRMATION}'"
    )

print("Install dependencies:", INSTALL_DEPENDENCIES)
print("Prepare Celeb-DF pilot:", PREPARE_CELEBDF_PILOT)
print("Audit Celeb-DF pilot:", AUDIT_CELEBDF_PILOT)
print("Force rebuild:", FORCE_REBUILD)
print("Seed:", SEED)
print("Pilot videos: 20 real + 20 fake")
print("Frames per video:", FRAMES_PER_VIDEO)
print("Face margin:", FACE_MARGIN_RATIO)
print("Saved face size:", f"{FACE_SIZE}x{FACE_SIZE}")


Install dependencies: True
Prepare Celeb-DF pilot: True
Audit Celeb-DF pilot: True
Force rebuild: False
Seed: 42
Pilot videos: 20 real + 20 fake
Frames per video: 10
Face margin: 0.15
Saved face size: 224x224


## 2. Environment and project directories


In [2]:
import json
import random
import re
import subprocess
import sys
import time
from pathlib import Path

if INSTALL_DEPENDENCIES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "kagglehub", "opencv-python", "pandas", "Pillow", "retina-face",
    ])

from google.colab import drive
drive.mount("/content/drive")

import cv2
import kagglehub
import numpy as np
import pandas as pd
from PIL import Image

BASE_PATH = Path("/content/drive/MyDrive/deepfake_project")
CELEBDF_DIR = BASE_PATH / "celebdf_v2"
PILOT_FRAMES_DIR = CELEBDF_DIR / "pilot_frames"
PILOT_FACES_DIR = CELEBDF_DIR / "pilot_faces"
MANIFEST_PATH = CELEBDF_DIR / "celebdf_pilot_manifest.json"
FRAME_LOG_PATH = CELEBDF_DIR / "celebdf_pilot_frame_extraction_log.csv"
FACE_LOG_PATH = CELEBDF_DIR / "celebdf_pilot_face_detection_log.csv"
RESULTS_DIR = BASE_PATH / "results" / "cross_dataset"
PLOTS_DIR = BASE_PATH / "plots" / "cross_dataset"

for folder in [
    PILOT_FRAMES_DIR / "real", PILOT_FRAMES_DIR / "fake",
    PILOT_FACES_DIR / "real", PILOT_FACES_DIR / "fake",
    RESULTS_DIR, PLOTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_PATH)
print("Celeb-DF pilot root:", CELEBDF_DIR)
print("Manifest:", MANIFEST_PATH)


Mounted at /content/drive
Project root: /content/drive/MyDrive/deepfake_project
Celeb-DF pilot root: /content/drive/MyDrive/deepfake_project/celebdf_v2
Manifest: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_manifest.json


## 3. Safe forced rebuild

This cell removes only generated pilot manifests, logs, frames and face crops—and only when the two rebuild controls have been deliberately enabled. It never deletes the downloaded Kaggle dataset.


In [3]:
GENERATED_FOLDERS = [
    PILOT_FRAMES_DIR / "real", PILOT_FRAMES_DIR / "fake",
    PILOT_FACES_DIR / "real", PILOT_FACES_DIR / "fake",
]
GENERATED_FILES = [MANIFEST_PATH, FRAME_LOG_PATH, FACE_LOG_PATH]

if FORCE_REBUILD:
    removed = 0
    for folder in GENERATED_FOLDERS:
        for path in folder.iterdir():
            if path.is_file():
                path.unlink()
                removed += 1
    for path in GENERATED_FILES:
        if path.exists():
            path.unlink()
            removed += 1
    print("Forced rebuild enabled. Generated pilot items removed:", removed)
else:
    print("Safe mode: existing manifest and generated pilot files are preserved.")


Safe mode: existing manifest and generated pilot files are preserved.


## 4. Download and discover Celeb-DF source videos

KaggleHub stores the full source dataset in the temporary Colab/Kaggle cache. Google Drive stores only the fixed 40-video manifest and the extracted 400-frame pilot.


In [4]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

def discover_source_videos(download_root):
    videos = sorted(
        path for path in Path(download_root).rglob("*")
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )
    real, fake = [], []
    for path in videos:
        parts = [part.lower() for part in path.parts]
        if any("synthesis" in part for part in parts):
            fake.append(path)
        elif any(part in {"celeb-real", "youtube-real"} for part in parts):
            real.append(path)
    return real, fake

download_root = None
source_real_videos = []
source_fake_videos = []

if PREPARE_CELEBDF_PILOT and not MANIFEST_PATH.exists():
    download_root = Path(kagglehub.dataset_download("reubensuju/celeb-df-v2"))
    source_real_videos, source_fake_videos = discover_source_videos(download_root)
    assert len(source_real_videos) >= PILOT_REAL_VIDEOS, "Not enough real source videos found."
    assert len(source_fake_videos) >= PILOT_FAKE_VIDEOS, "Not enough fake source videos found."
    display(pd.DataFrame([
        {"class": "real", "source_videos_found": len(source_real_videos)},
        {"class": "fake", "source_videos_found": len(source_fake_videos)},
    ]))
elif MANIFEST_PATH.exists():
    print("Saved manifest exists; source dataset download is not required unless frames are missing.")
else:
    print("Preparation disabled; source dataset was not downloaded.")


100%|██████████| 9.29G/9.29G [10:23<00:00, 16.0MB/s]

Extracting files...


,class,source_videos_found
0,real,890
1,fake,5639


## 5. Create or load the fixed pilot manifest

The first build selects videos deterministically with seed 42 and saves their paths relative to the Kaggle dataset root. Every subsequent run loads the same manifest instead of selecting videos again.


In [5]:
def safe_slug(value):
    value = re.sub(r"[^A-Za-z0-9_-]+", "_", value).strip("_")
    return value or "video"

if PREPARE_CELEBDF_PILOT:
    existing_generated_images = sum(
        len([p for p in folder.iterdir() if p.is_file()])
        for folder in GENERATED_FOLDERS
    )

    if MANIFEST_PATH.exists():
        with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
            pilot_manifest = json.load(handle)
        print("Loaded existing pilot manifest:", MANIFEST_PATH)
    else:
        if existing_generated_images:
            raise RuntimeError(
                "Generated pilot images exist without a manifest. Preserve them and investigate, "
                "or explicitly use the protected FORCE_REBUILD controls."
            )
        rng = random.Random(SEED)
        selected_real = rng.sample(source_real_videos, PILOT_REAL_VIDEOS)
        selected_fake = rng.sample(source_fake_videos, PILOT_FAKE_VIDEOS)

        def records(paths, class_name):
            result = []
            for index, path in enumerate(paths):
                result.append({
                    "class": class_name,
                    "video_id": f"{class_name}__{index:03d}__{safe_slug(path.stem)}",
                    "relative_source_path": str(path.relative_to(download_root)),
                    "source_filename": path.name,
                })
            return result

        pilot_manifest = {
            "dataset": "Celeb-DF v2",
            "kaggle_dataset": "reubensuju/celeb-df-v2",
            "seed": SEED,
            "frames_per_video": FRAMES_PER_VIDEO,
            "face_margin_ratio": FACE_MARGIN_RATIO,
            "face_size": FACE_SIZE,
            "real_videos": records(selected_real, "real"),
            "fake_videos": records(selected_fake, "fake"),
        }
        with MANIFEST_PATH.open("w", encoding="utf-8") as handle:
            json.dump(pilot_manifest, handle, indent=2)
        print("Created fixed pilot manifest:", MANIFEST_PATH)

    assert pilot_manifest["seed"] == SEED
    assert pilot_manifest["frames_per_video"] == FRAMES_PER_VIDEO
    assert len(pilot_manifest["real_videos"]) == PILOT_REAL_VIDEOS
    assert len(pilot_manifest["fake_videos"]) == PILOT_FAKE_VIDEOS
    display(pd.DataFrame([
        {"class": "real", "selected_videos": len(pilot_manifest["real_videos"])},
        {"class": "fake", "selected_videos": len(pilot_manifest["fake_videos"])},
    ]))


Created fixed pilot manifest: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_manifest.json


,class,selected_videos
0,real,20
1,fake,20


## 6. Extract 10 deterministic frames per selected video

Existing expected frame files are skipped. If any frame is missing, only that frame is recreated from the source video selected in the saved manifest.


In [6]:
def expected_frame_path(record, frame_number):
    return PILOT_FRAMES_DIR / record["class"] / f"{record['video_id']}__frame_{frame_number:02d}.jpg"

def source_dataset_required():
    for class_key in ("real_videos", "fake_videos"):
        for record in pilot_manifest[class_key]:
            if any(not expected_frame_path(record, i).exists() for i in range(FRAMES_PER_VIDEO)):
                return True
    return False

if PREPARE_CELEBDF_PILOT and source_dataset_required() and download_root is None:
    download_root = Path(kagglehub.dataset_download("reubensuju/celeb-df-v2"))

def resolve_source_video(record):
    candidate = download_root / record["relative_source_path"]
    if candidate.exists():
        return candidate
    matches = list(download_root.rglob(record["source_filename"]))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Could not uniquely resolve source video: {record['source_filename']}"
        )
    return matches[0]

frame_log = []
if PREPARE_CELEBDF_PILOT:
    for class_key in ("real_videos", "fake_videos"):
        for record in pilot_manifest[class_key]:
            outputs = [expected_frame_path(record, i) for i in range(FRAMES_PER_VIDEO)]
            if all(path.exists() for path in outputs):
                frame_log.append({**record, "status": "already complete", "frames": FRAMES_PER_VIDEO})
                continue

            source_path = resolve_source_video(record)
            capture = cv2.VideoCapture(str(source_path))
            frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            if frame_count < FRAMES_PER_VIDEO:
                capture.release()
                raise RuntimeError(f"Too few frames in source video: {source_path}")

            indices = np.linspace(0, frame_count - 1, FRAMES_PER_VIDEO, dtype=int)
            saved = 0
            for output_number, source_index in enumerate(indices):
                output_path = outputs[output_number]
                if output_path.exists():
                    saved += 1
                    continue
                capture.set(cv2.CAP_PROP_POS_FRAMES, int(source_index))
                success, frame = capture.read()
                if not success or not cv2.imwrite(str(output_path), frame):
                    capture.release()
                    raise RuntimeError(f"Failed to extract frame {source_index} from {source_path}")
                saved += 1
            capture.release()
            frame_log.append({**record, "status": "complete", "frames": saved})

    pd.DataFrame(frame_log).to_csv(FRAME_LOG_PATH, index=False)
    display(pd.DataFrame(frame_log).groupby(["class", "status"])["frames"].sum())
    print("Frame extraction log:", FRAME_LOG_PATH)


,,frames
class,status,
fake,complete,200
real,complete,200


Frame extraction log: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_frame_extraction_log.csv


## 7. Create RetinaFace crops with a 15% margin

The largest detected face is expanded by 15% of the detected box width and height on each side, clipped to the image boundary, resized to 224×224 and saved with the same frame filename. Existing crops are skipped.


In [7]:
if PREPARE_CELEBDF_PILOT:
    from retinaface import RetinaFace

def largest_face_box(detections):
    if not isinstance(detections, dict):
        return None
    boxes = []
    for detection in detections.values():
        area = detection.get("facial_area")
        if area is not None and len(area) == 4:
            x1, y1, x2, y2 = map(int, area)
            boxes.append((x1, y1, x2, y2))
    return max(boxes, key=lambda b: max(0, b[2]-b[0]) * max(0, b[3]-b[1])) if boxes else None

face_log = []
if PREPARE_CELEBDF_PILOT:
    started = time.time()
    for class_name in ("real", "fake"):
        frame_paths = sorted(
            p for p in (PILOT_FRAMES_DIR / class_name).iterdir()
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        for index, frame_path in enumerate(frame_paths, start=1):
            output_path = PILOT_FACES_DIR / class_name / frame_path.name
            if output_path.exists():
                face_log.append({"class": class_name, "filename": frame_path.name, "status": "already existed"})
                continue

            image_bgr = cv2.imread(str(frame_path))
            if image_bgr is None:
                raise RuntimeError(f"Could not read frame: {frame_path}")
            detections = RetinaFace.detect_faces(str(frame_path))
            box = largest_face_box(detections)
            if box is None:
                face_log.append({"class": class_name, "filename": frame_path.name, "status": "no face detected"})
                continue

            x1, y1, x2, y2 = box
            height, width = image_bgr.shape[:2]
            margin_x = int((x2 - x1) * FACE_MARGIN_RATIO)
            margin_y = int((y2 - y1) * FACE_MARGIN_RATIO)
            x1, x2 = max(0, x1-margin_x), min(width, x2+margin_x)
            y1, y2 = max(0, y1-margin_y), min(height, y2+margin_y)
            crop_bgr = image_bgr[y1:y2, x1:x2]
            if crop_bgr.size == 0:
                face_log.append({"class": class_name, "filename": frame_path.name, "status": "invalid crop"})
                continue
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
            Image.fromarray(crop_rgb).resize(
                (FACE_SIZE, FACE_SIZE), Image.Resampling.LANCZOS
            ).save(output_path, quality=95)
            face_log.append({"class": class_name, "filename": frame_path.name, "status": "saved"})

            if index % 100 == 0:
                print(f"{class_name}: {index}/{len(frame_paths)}")

    pd.DataFrame(face_log).to_csv(FACE_LOG_PATH, index=False)
    print(f"Face preprocessing completed in {(time.time()-started)/60:.2f} minutes.")
    display(pd.DataFrame(face_log).groupby(["class", "status"]).size().rename("images"))
    print("Face detection log:", FACE_LOG_PATH)


26-08-11 14:37:30 - Directory /root/.deepface created
26-08-11 14:37:30 - Directory /root/.deepface/weights created
26-08-11 14:37:30 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5
100%|██████████| 119M/119M [00:00<00:00, 427MB/s] 


real: 100/200
real: 200/200
fake: 100/200
fake: 200/200
Face preprocessing completed in 2.14 minutes.


,,images
class,status,
fake,saved,200
real,saved,200


Face detection log: /content/drive/MyDrive/deepfake_project/celebdf_v2/celebdf_pilot_face_detection_log.csv


## 8. Final pilot audit

The build is accepted only when both representations contain exactly 200 real and 200 fake images, recover 20 source videos per class and contain 10 images per video.


In [8]:
VALID_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(folder):
    return sorted(
        p for p in Path(folder).iterdir()
        if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
    )

def video_id_from_filename(path):
    if "__frame_" not in path.stem:
        raise ValueError(f"Unexpected pilot filename: {path.name}")
    return path.stem.split("__frame_", 1)[0]

if AUDIT_CELEBDF_PILOT:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError("Pilot manifest has not been created.")
    with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        saved_manifest = json.load(handle)

    rows = []
    for representation, root in [("full_frame", PILOT_FRAMES_DIR), ("face_crop", PILOT_FACES_DIR)]:
        for class_name in ("real", "fake"):
            paths = image_files(root / class_name)
            counts = {}
            for path in paths:
                video_id = video_id_from_filename(path)
                counts[video_id] = counts.get(video_id, 0) + 1
            rows.append({
                "representation": representation,
                "class": class_name,
                "images": len(paths),
                "source_videos": len(counts),
                "minimum_per_video": min(counts.values()) if counts else 0,
                "maximum_per_video": max(counts.values()) if counts else 0,
            })
            assert len(paths) == 200, f"{representation}/{class_name}: expected 200 images"
            assert len(counts) == 20, f"{representation}/{class_name}: expected 20 videos"
            assert set(counts.values()) == {10}, f"{representation}/{class_name}: expected 10 images per video"

    assert saved_manifest.get("face_margin_ratio") == FACE_MARGIN_RATIO
    assert saved_manifest.get("face_size") == FACE_SIZE
    display(pd.DataFrame(rows))
    print("PASS: Celeb-DF pilot is ready for the separate Section 5 evaluators.")


,representation,class,images,source_videos,minimum_per_video,maximum_per_video
0,full_frame,real,200,20,10,10
1,full_frame,fake,200,20,10,10
2,face_crop,real,200,20,10,10
3,face_crop,fake,200,20,10,10


PASS: Celeb-DF pilot is ready for the separate Section 5 evaluators.


## 9. Subsequent reruns

After the first successful build:

```python
INSTALL_DEPENDENCIES = False
PREPARE_CELEBDF_PILOT = True
AUDIT_CELEBDF_PILOT = True
FORCE_REBUILD = False
```

The existing manifest is loaded, completed frames and face crops are skipped, and the final audit is repeated. Do not enable `FORCE_REBUILD` unless you deliberately want a different fresh pilot build.


## 10. Next stage

Once the final audit passes, use separate notebooks for actual inference, beginning with `05a_E1_CelebDF_Evaluation.ipynb`. Each evaluator must validate its required input, load one fixed FF++ checkpoint and save frame/video results independently.
